<a href="https://colab.research.google.com/github/serjtankian/practica-keepcoding-NLP-Search/blob/main/NLP_Preprocesado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import pandas as pd
import en_core_web_sm
import unicodedata
from spacy.lang.en.stop_words import STOP_WORDS



In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
ruta_real = '/content/drive/MyDrive/reviews_Electronics_5.json.gz'
df = pd.read_json(ruta_real, lines=True)

In [ ]:
df.head()

,reviewerID,asin,reviewerName,helpful,reviewText,overall,summary,unixReviewTime,reviewTime
0,AO94DHGC771SJ,0528881469,amazdnu,"[0, 0]",We got this GPS for my husband who is an (OTR)...,5,Gotta have GPS!,1370131200,"06 2, 2013"
1,AMO214LNFCEI4,0528881469,Amazon Customer,"[12, 15]","I'm a professional OTR truck driver, and I bou...",1,Very Disappointed,1290643200,"11 25, 2010"
2,A3N7T0DY83Y4IG,0528881469,C. A. Freeman,"[43, 45]","Well, what can I say. I've had this unit in m...",3,1st impression,1283990400,"09 9, 2010"
3,A1H8PY3QHMQQA0,0528881469,"Dave M. Shaw ""mack dave""","[9, 10]","Not going to write a long review, even thought...",2,"Great grafics, POOR GPS",1290556800,"11 24, 2010"
4,A24EV6RXELQZ63,0528881469,Wayne Smith,"[0, 0]",I've had mine for a year and here's what we go...,1,"Major issues, only excuses for support",1317254400,"09 29, 2011"


In [ ]:
!pip install num2words

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 6.0 MB/s eta 0:00:00
  Created wheel for docopt: filename=docopt-0.6.2-py2.py3-none-any.whl size=13706 sha256=d400c31508448ab53589ddfc559888b481bdd06f30a7b3ce385e5e1ba57eecf6
  Stored in directory: /root/.cache/pip/wheels/1a/bf/a1/4cee4f7678c68c5875ca89eaccf460593539805c3906722228
Successfully built docopt


In [ ]:
df = df.sample(5000, random_state=42).reset_index(drop=True)
print(f"Trabajando con {len(df)} reviews")

nlp_en = en_core_web_sm.load()

SW = STOP_WORDS - {'not', 'no', 'never'}

def preprocess_text(text):
    if not isinstance(text, str):
        return ''

    text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8', 'ignore')
    text = text.lower()

    doc = nlp_en(text)

    tokens = []
    for token in doc:
        if token.text in SW:
            continue
        if token.is_punct:
            continue
        tokens.append(token.lemma_)

    return ' '.join(tokens)

Trabajando con 5000 reviews


In [ ]:
!pip install langdetect
from langdetect import detect

def detectar_idioma(texto):
    try:
        return detect(texto)
    except:
        return "unknown"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 16.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=8f2fdd4da5ccdcfc66f239f6069abe34ff44fe610547a5b429c91f6ec4dfa6a3
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect


In [ ]:


df['idioma'] = df['reviewText'].fillna('').apply(detectar_idioma)

df = df[df['idioma'] == 'en'].reset_index(drop=True)

In [ ]:
df['text_clean'] = df['reviewText'].fillna('').apply(preprocess_text)

print(df[['reviewText', 'text_clean']].head())

                                          reviewText  \
0  Well, after trying out some Box Towers....that...   
1  I ordered one for my wife and one for me. Afte...   
2  The sound quality of this unit is phenomenal. ...   
3  It is good on keeping your cpu cool also down'...   
4  Well made, priced right & works like it should...   

                                          text_clean  
0  try box tower sound like music box try magnepa...  
1  order wife read negative review picture qualit...  
2  sound quality unit phenomenal   bose sounddock...  
3  good keep cpu cool down't forget download cors...  
4  price right work like ask mount directly ceili...  


In [ ]:
df['sentiment_label'] = df['overall'].apply(lambda x: 1 if x >= 4 else 0)

df.to_csv('/content/drive/MyDrive/electronics_preprocessed.csv', index=False)
print(f"Guardado: {df.shape[0]} filas, columnas: {list(df.columns)}")

Guardado: 4982 filas, columnas: ['reviewerID', 'asin', 'reviewerName', 'helpful', 'reviewText', 'overall', 'summary', 'unixReviewTime', 'reviewTime', 'texto_limpio', 'idioma', 'text_clean', 'sentiment_label']


**Observaciones**



*   Se selecciona una muestra de 5 mil reviws para evitar problemas de memoria.
*   Se aplica deteccion de idioma para eliminar reviews en español.


*   Se elimina puntuacion y numeros y se normaliza la muestra



